In [1]:
import sys
from pathlib import Path

import os
import requests
from dotenv import load_dotenv

import warnings
# on va ignorer les messages avertissements a cause d'un RAGASS pas compatible mystral v2.0
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [2]:
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

load_dotenv()

TOKEN = os.getenv("API_KEY")
TOKEN_MISTRAL = os.getenv("MISTRAL_API_KEY")
TOKEN_OPENAGENDA = os.getenv("OPENAGENDA_API_KEY")

API_URL_ASK = "http://127.0.0.1:8000/ask"
API_URL_ASK_DEBUG = "http://127.0.0.1:8000/ask/debug"


In [3]:
import requests
from typing import Any


def call_ask_api(question: str) -> dict[str, Any]:
    """
    Interroge l'endpoint /ask de l'API RAG.

    Parameters
    ----------
    question : str
        Question utilisateur à envoyer à l'API.

    Returns
    -------
    dict[str, Any]
        Réponse JSON de l'API.
    """
    resp = requests.post(
        API_URL_ASK,
        json={"question": question},
        headers={"x-api-key": TOKEN},
        timeout=60,
    )

    print("STATUS /ask :", resp.status_code)
    data = resp.json()
    print("ANSWER /ask :")
    print(data.get("answer", ""))

    return data


def run_rag_for_eval(question: str) -> dict[str, Any]:
    """
    Interroge l'endpoint /ask_debug pour récupérer la réponse
    ainsi que les contextes utilisés par le pipeline RAG.

    Parameters
    ----------
    question : str
        Question d'évaluation.

    Returns
    -------
    dict[str, Any]
        Dictionnaire contenant :
        - question
        - answer
        - contexts
        - error éventuel
    """
    resp = requests.post(
        API_URL_ASK_DEBUG,
        json={"question": question},
        headers={"x-api-key": TOKEN},
        timeout=60,
    )

    print(f"QUESTION : {question}")
    print("STATUS /ask_debug :", resp.status_code)

    try:
        data = resp.json()
    except Exception:
        data = {"raw_text": resp.text}

    if resp.status_code != 200:
        print("ERROR :", data)
        return {
            "question": question,
            "error": data,
            "answer": None,
            "contexts": None,
        }

    return {
        "question": question,
        "answer": data.get("answer"),
        "contexts": data.get("contexts"),
        "error": None,
    }


# ------------------------------------------------------------------
# Test rapide manuel de l'API
# ------------------------------------------------------------------
sample_question = "Quels événements culturels gratuits ont lieu à Montpellier ?"
sample_response = call_ask_api(sample_question)

STATUS /ask : 200
ANSWER /ask :
Je ne trouve pas d'événement correspondant, je suis un assistant qui ne peut vous conseiller que des événements culturels.


In [4]:
import requests
from typing import Any


def call_ask_api(question: str) -> dict[str, Any]:
    """
    Interroge l'endpoint /ask de l'API RAG.

    Parameters
    ----------
    question : str
        Question utilisateur à envoyer à l'API.

    Returns
    -------
    dict[str, Any]
        Réponse JSON de l'API.
    """
    resp = requests.post(
        API_URL_ASK,
        json={"question": question},
        headers={"x-api-key": TOKEN},
        timeout=60,
    )

    print("STATUS /ask :", resp.status_code)
    data = resp.json()
    print("ANSWER /ask :")
    print(data.get("answer", ""))

    return data


def run_rag_for_eval(question: str) -> dict[str, Any]:
    """
    Interroge l'endpoint /ask_debug pour récupérer la réponse
    ainsi que les contextes utilisés par le pipeline RAG.

    Parameters
    ----------
    question : str
        Question d'évaluation.

    Returns
    -------
    dict[str, Any]
        Dictionnaire contenant :
        - question
        - answer
        - contexts
        - error éventuel
    """
    resp = requests.post(
        API_URL_ASK_DEBUG,
        json={"question": question},
        headers={"x-api-key": TOKEN},
        timeout=60,
    )

    print(f"QUESTION : {question}")
    print("STATUS /ask_debug :", resp.status_code)

    try:
        data = resp.json()
    except Exception:
        data = {"raw_text": resp.text}

    if resp.status_code != 200:
        print("ERROR :", data)
        return {
            "question": question,
            "error": data,
            "answer": None,
            "contexts": None,
        }

    return {
        "question": question,
        "answer": data.get("answer"),
        "contexts": data.get("contexts"),
        "error": None,
    }


# ------------------------------------------------------------------
# Test rapide manuel de l'API
# ------------------------------------------------------------------
sample_question = "Quels événements culturels gratuits ont lieu à Montpellier ?"
sample_response = call_ask_api(sample_question)

STATUS /ask : 200
ANSWER /ask :
Je ne trouve pas d'événement correspondant, je suis un assistant qui ne peut vous conseiller que des événements culturels.


In [ ]:
eval_questions_positive = [
{
    "question": "Y a-t-il des expositions à Montpellier ?",
    "ground_truth": (
        "Expositions à Montpellier :\n"
        "- L'association Miss'terre fait son expo d'été "
        "(date: le 27 et le 28 juin 2025, lieu : l'Atelier Miss'terre).\n" #vu
        "- Expositions 'Montpellier, regarder la ville autrement. Archives et architecture' "
        "(date : 20 septembre 2025, lieu : les Archives de Montpellier).\n" #vu
        "- Exposition 'L'Europe et son patrimoine' "
        "(date : 20 et 21 septembre 2025, lieu : la Maison de l'Europe de Montpellier)." #vu
    ),
    "group": "positive",
},
{
    "question": "Quels événements ont lieu au Musée Fabre ?",
    "ground_truth": (
        "dans le lieu le Musée Fabre à Montpellier, un événement est proposé :\n"
        "- Visite libre de la bibliothèque Jean Claparède "
        "(date : 20 et 21 septembre 2025, lieu : le Musée Fabre)." #vu
    ),
    "group": "positive",
},
{
    "question": "Quels événements ont lieu à Montpellier en septembre 2025 ?",
    "ground_truth": (
        "Voici quelques événements organisés à Montpellier en septembre 2025 :\n"
        "- Conférences, expositions et dédicaces au Cercle Culturel Languedocien "
        "(date : 20 et 21 septembre 2025, lieu : Le Cercle Culturel Languedocien).\n" #vu
        "- Expositions 'Montpellier, regarder la ville autrement. Archives et architecture' "
        "(date : 20 septembre 2025, lieu : les Archives de Montpellier).\n" #vu
        "- Exposition 'L'Europe et son patrimoine' "
        "(date : 20 et 21 septembre 2025, lieu : la Maison de l'Europe de Montpellier)." #vu
    ),
    "group": "positive",
},
{
    "question": "Quels événements ont lieu aux Archives de Montpellier ?",
    "ground_truth": (
        "Aux Archives de Montpellier, un événement est organisé :\n"
        "- 'Montpellier, regarder la ville autrement. Archives et architecture' "
        "(20 septembre 2025)."
    ),
    "group": "positive",
},
{
    "question": "Quels événements ont lieu le 20 septembre 2025 à Montpellier ?",
    "ground_truth": (
        "Voici quelques événements organisés à Montpellier le 20 septembre 2025 :\n"
        "- Visite libre de la bibliothèque Jean Claparède (Musée Fabre).\n"
        "- Conférences, expositions et dédicaces au Cercle Culturel Languedocien.\n"
        "- Exposition 'Montpellier, regarder la ville autrement. Archives et architecture' "
        "(Archives de Montpellier)."
    ),
    "group": "positive",
},
{
    "question": "Quels événements ont lieu le week-end du 20 au 21 septembre 2025 à Montpellier ?",
    "ground_truth": (
        "À Montpellier, plusieurs événements ont lieu les 20 et 21 septembre 2025 :\n"
        "- Visite libre de la bibliothèque Jean Claparède (Musée Fabre).\n"
        "- Conférences, expositions et dédicaces au Cercle Culturel Languedocien.\n"
        "- 'L'Europe et son patrimoine' (20-21 septembre, Maison de l'Europe de Montpellier).\n"
        "- 'Montpellier, regarder la ville autrement. Archives et architecture' "
        "(20 septembre, Archives de Montpellier)."
    ),
    "group": "positive",
}
]

eval_questions_ambiguous = [
{
    "question": "Quels événements culturels sont proposés à Montpellier ?",
    "ground_truth": (
        "Voici quelques événements culturels proposés à Montpellier :\n"
        "- Visite libre de la bibliothèque Jean Claparède (Musée Fabre).\n"
        "- Conférences, expositions et dédicaces au Cercle Culturel Languedocien.\n"
        "- Exposition 'Montpellier, regarder la ville autrement. Archives et architecture' "
        "(Archives de Montpellier).\n"
        "- Exposition 'L'Europe et son patrimoine' "
        "(Maison de l'Europe de Montpellier)."
    ),
    "group": "ambiguous",
},
    {
        "question": "Quels événements gratuits ont lieu à Montpellier ?",
        "ground_truth": (
            "Le corpus ne permet pas de confirmer explicitement la gratuité des événements, "
            "aucun événement ne peut donc être identifié avec certitude comme gratuit."
        ),
        "group": "ambiguous",
    }
]

eval_questions_negative = [
{
    "question": "Y a-t-il des concerts de rock à Montpellier?",
    "ground_truth": (
        "Je n'ai trouvé aucun concert de rock à Montpellier dans les données disponibles."
    ),
    "group": "negative",
},
{
    "question": "Quels événements ont lieu à Paris ?",
    "ground_truth": (
        "Je n'ai trouvé aucun événement correspondant à Paris dans les données disponibles."
    ),
    "group": "negative",
}
]

eval_questions = (
    eval_questions_positive
    + eval_questions_ambiguous
    + eval_questions_negative
)

len(eval_questions)

10

In [6]:
#construction dataset evaluation
rows = []

for item in eval_questions:
    result = run_rag_for_eval(item["question"])

    rows.append(
        {
            "question": item["question"],
            "answer": result["answer"],
            "contexts": result["contexts"],
            "ground_truth": item["ground_truth"],
            "group": item["group"],
            "error": result["error"],
        }
    )

rows[:1]

QUESTION : Y a-t-il des expositions à Montpellier ?
STATUS /ask_debug : 200
QUESTION : Quels événements ont lieu au Musée Fabre ?
STATUS /ask_debug : 200
QUESTION : Quels événements ont lieu à Montpellier en septembre 2025 ?
STATUS /ask_debug : 200
QUESTION : Quels événements ont lieu aux Archives de Montpellier ?
STATUS /ask_debug : 200
QUESTION : Quels événements ont lieu le 20 septembre 2025 à Montpellier ?
STATUS /ask_debug : 200
QUESTION : Quels événements ont lieu le week-end du 20 au 21 septembre 2025 à Montpellier ?
STATUS /ask_debug : 200
QUESTION : Quels événements culturels sont proposés à Montpellier ?
STATUS /ask_debug : 200
QUESTION : Quels événements gratuits ont lieu à Montpellier ?
STATUS /ask_debug : 200
QUESTION : Y a-t-il des concerts de rock à Montpellier?
STATUS /ask_debug : 200
QUESTION : Quels événements ont lieu à Paris ?
STATUS /ask_debug : 200


[{'question': 'Y a-t-il des expositions à Montpellier ?',
  'answer': 'Voici les expositions à Montpellier disponibles dans le contexte documentaire :\n\n---\n\n**Titre :** L\'association Miss\'terre fait son expo d\'été !\n\n**Lieu :** Association Miss\'terre, Montpellier\n**Date :** 27 juin 2025 – 28 juin 2025\n**Description :** Exposition organisée par l\'association Miss\'terre dans le cadre de son événement estival.\n\n**Pourquoi cet événement pourrait vous intéresser :**\nSi vous aimez les expositions associatives et les événements locaux, cette expo d\'été pourrait vous plaire, avec une ambiance conviviale et engagée.\n\n---\n\n**Titre :** Exposition "Montpellier, regarder la ville autrement. Archives et architecture"\n\n**Lieu :** Archives de Montpellier\n**Date :** 20 septembre 2025\n**Description :** Découverte de l\'évolution architecturale de Montpellier à travers des documents d\'archives et cinq monuments emblématiques.\n\n**Pourquoi cet événement pourrait vous intéresser

In [7]:
import pandas as pd

df_eval_raw = pd.DataFrame(rows)

print(df_eval_raw.shape)
df_eval_raw[["question", "group", "error"]]

(10, 6)


,question,group,error
0,Y a-t-il des expositions à Montpellier ?,positive,None
1,Quels événements ont lieu au Musée Fabre ?,positive,None
2,Quels événements ont lieu à Montpellier en sep...,positive,None
3,Quels événements ont lieu aux Archives de Mont...,positive,None
4,Quels événements ont lieu le 20 septembre 2025...,positive,None
5,Quels événements ont lieu le week-end du 20 au...,positive,None
6,Quels événements culturels sont proposés à Mon...,ambiguous,None
7,Quels événements gratuits ont lieu à Montpelli...,ambiguous,None
8,Y a-t-il des concerts de rock à Montpellier?,negative,None
9,Quels événements ont lieu à Paris ?,negative,None


In [8]:
for i, row in enumerate(rows):
    assert isinstance(row["question"], str), f"question row {i} invalide"
    assert isinstance(row["ground_truth"], str), f"ground_truth row {i} invalide"

    if row["answer"] is not None:
        assert isinstance(row["answer"], str), f"answer row {i} invalide"

    if row["contexts"] is not None:
        assert isinstance(row["contexts"], list), f"contexts row {i} invalide"
        assert all(isinstance(c, str) for c in row["contexts"]), (
            f"contexts row {i} contient autre chose que str"
        )

print("Format OK pour les rows")

Format OK pour les rows


In [9]:
from ragas.dataset_schema import EvaluationDataset

ragas_rows = [
    {
        "user_input": row["question"],
        "response": row["answer"],
        "retrieved_contexts": row["contexts"],
        "reference": row["ground_truth"],
        "group": row["group"],
    }
    for row in rows
    if row["answer"] is not None and row["contexts"] is not None
]

ragas_dataset = EvaluationDataset.from_list(ragas_rows)

print(ragas_rows[0])
print(ragas_dataset)

{'user_input': 'Y a-t-il des expositions à Montpellier ?', 'response': 'Voici les expositions à Montpellier disponibles dans le contexte documentaire :\n\n---\n\n**Titre :** L\'association Miss\'terre fait son expo d\'été !\n\n**Lieu :** Association Miss\'terre, Montpellier\n**Date :** 27 juin 2025 – 28 juin 2025\n**Description :** Exposition organisée par l\'association Miss\'terre dans le cadre de son événement estival.\n\n**Pourquoi cet événement pourrait vous intéresser :**\nSi vous aimez les expositions associatives et les événements locaux, cette expo d\'été pourrait vous plaire, avec une ambiance conviviale et engagée.\n\n---\n\n**Titre :** Exposition "Montpellier, regarder la ville autrement. Archives et architecture"\n\n**Lieu :** Archives de Montpellier\n**Date :** 20 septembre 2025\n**Description :** Découverte de l\'évolution architecturale de Montpellier à travers des documents d\'archives et cinq monuments emblématiques.\n\n**Pourquoi cet événement pourrait vous intéresse

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_mistralai import ChatMistralAI

from ragas import evaluate
from ragas.dataset_schema import EvaluationDataset
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    answer_relevancy,
    context_precision,
    context_recall,
    faithfulness,
)

llm = ChatMistralAI(
    model="mistral-small-latest",
    api_key=TOKEN_MISTRAL.strip(),
    temperature=0,
)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

small_ragas_dataset = EvaluationDataset.from_list(ragas_rows[:10])

result = evaluate(
    dataset=small_ragas_dataset,
    metrics=[
        faithfulness,
        context_precision,
        context_recall,
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    raise_exceptions=False,
)

print(result)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

Exception raised in Job[6]: OutputParserException(Failed to parse StringIO from completion {"statements": [{"statement": "Montpellier will host several events in September 2025.", "reason": "The context lists multiple events in Montpellier in September 2025, including guided tours and a ball dance.", "verdict": 1}, {"statement": "One event is a guided tour of the Protestant Cemetery of Montpellier.", "reason": "The context explicitly mentions a guided tour of the Protestant Cemetery of Montpellier.", "verdict": 1}, {"statement": "The Protestant Cemetery of Montpellier is located in Montpellier.", "reason": "The context states that the Protestant Cemetery of Montpellier is in Montpellier.", "verdict": 1}, {"statement": "The guided tour of the Protestant Cemetery of Montpellier will take place on September 20 and 21, 2025.", "reason": "The context specifies the dates for the guided tour of the Protestant Cemetery as September 20 and 21, 2025.", "verdict": 1}, {"statement": "The guided to

{'faithfulness': 0.5855, 'context_precision': 0.5750, 'context_recall': 0.5850}


In [11]:
result_df = result.to_pandas()
result_df

,user_input,retrieved_contexts,response,reference,faithfulness,context_precision,context_recall
0,Y a-t-il des expositions à Montpellier ?,[L'association Miss'terre fait son expo d'été ...,Voici les expositions à Montpellier disponible...,"Oui, plusieurs expositions ont lieu à Montpell...",0.933333,0.583333,0.75
1,Quels événements ont lieu au Musée Fabre ?,"[Visite commentée : « Correspondances, Max Rou...",Voici les événements ayant lieu au Musée Fabre...,"Au Musée Fabre à Montpellier, un événement est...",0.526316,0.500000,0.00
2,Quels événements ont lieu à Montpellier en sep...,[Visite guidée du cimetière protestant de Mont...,Voici les événements à Montpellier en septembr...,Voici quelques événements organisés à Montpell...,NaN,0.000000,0.00
3,Quels événements ont lieu aux Archives de Mont...,"[Exposition ""Montpellier, regarder la ville au...",Voici les événements ayant lieu aux **Archives...,"Aux Archives de Montpellier, un événement est ...",1.000000,1.000000,1.00
4,Quels événements ont lieu le 20 septembre 2025...,[Visite guidée du cimetière protestant de Mont...,Titre : Visite guidée du cimetière protestant ...,Voici quelques événements organisés à Montpell...,1.000000,0.333333,0.50
5,Quels événements ont lieu le week-end du 20 au...,"[Week-end ""Éclosion"" - 4 pratiques transformat...",Titre : Visite guidée du cimetière protestant ...,"À Montpellier, plusieurs événements ont lieu l...",1.000000,0.000000,0.20
6,Quels événements culturels sont proposés à Mon...,[Visite guidée du cimetière protestant de Mont...,Voici les événements culturels proposés à Mont...,Voici quelques événements culturels proposés à...,0.809524,0.333333,0.40
7,Quels événements gratuits ont lieu à Montpelli...,[Visite guidée du cimetière protestant de Mont...,"Je ne trouve pas d'événement correspondant, je...",Le corpus ne permet pas de confirmer explicite...,0.000000,1.000000,1.00
8,Y a-t-il des concerts de rock à Montpellier?,[Concert de gospel\n🎤 Concert de gospel – Clôt...,"Je ne trouve pas d'événement correspondant, je...",Je n'ai trouvé aucun concert de rock à Montpel...,0.000000,1.000000,1.00
9,Quels événements ont lieu à Paris ?,[La Dame de Pierre\nDécouvrez le récit légenda...,"Je ne trouve pas d'événement correspondant, je...",Je n'ai trouvé aucun événement correspondant à...,0.000000,1.000000,1.00


In [12]:
print(result)

{'faithfulness': 0.5855, 'context_precision': 0.5750, 'context_recall': 0.5850}
